# 2D-FLCS Analysis Example

This notebook demonstrates how to use the 2D-FLCS computational core from Python, independent of the ChiSurf GUI.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tttrlib

# Add the plugin directory to path if needed
# sys.path.append('path/to/chisurf/plugins/fcs')
from chisurf.plugins.fcs.fcs_2d.api import correlate_tttr, fit_mem_2d

## 1. Load TTTR Data

We use `tttrlib` to load the time-tagged data.

In [ ]:
# Replace with your actual TTTR file path
file_path = "test_data.ptu"

if Path(file_path).exists():
    data = tttrlib.TTTR(file_path)
    header = data.get_header()
    macro_res = header.macro_time_resolution
    micro_res = header.micro_time_resolution
    print(f"Loaded {len(data)} photons")
    print(f"Macro resolution: {macro_res} s")
    print(f"Micro resolution: {micro_res} s")
else:
    print(f"File {file_path} not found. Using placeholder data for demonstration.")
    # Placeholder data generation
    n_photons = 100000
    macro_times = np.sort(np.random.randint(0, 1000000, n_photons).astype(np.int64))
    micro_times = np.random.randint(0, 4096, n_photons).astype(np.int64)
    macro_res = 1e-7
    micro_res = 1e-12

## 2. Generate 2D-FDC Matrices

Use the high-level `correlate_tttr` API.

In [ ]:
# Parameters in ticks (or physical units depending on implementation)
dT_ticks = int(100e-6 / macro_res)
ddT_ticks = int(50e-6 / macro_res)
tMin_ticks = int(1e-9 / micro_res)
tMax_ticks = int(12e-9 / micro_res)

result = correlate_tttr(
    macro_times=data.macro_times if "data" in locals() else macro_times,
    micro_times=data.micro_times if "data" in locals() else micro_times,
    dT=dT_ticks,
    ddT=ddT_ticks,
    tMin=tMin_ticks,
    tMax=tMax_ticks,
    logt_imax=100,
)

print("2D-FDC Matrix generated.")

## 3. Visualize Correlation Matrices

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

im1 = ax1.imshow(np.log10(result["mat_lin"] + 1), origin="lower", cmap="viridis")
ax1.set_title("Linear 2D-FDC (Log Intensity)")
ax1.set_xlabel("\u03c4\u2081 (bins)")
ax1.set_ylabel("\u03c4\u2082 (bins)")
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(np.log10(result["mat_log"] + 1), origin="lower", cmap="magma")
ax2.set_title("Log-Scale 2D-FDC (Log Intensity)")
ax2.set_xlabel("\u03c4\u2081 (log-points)")
ax2.set_ylabel("\u03c4\u2082 (log-points)")
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

## 4. Perform 2D-MEM Fitting

Now we fit the generated linear matrix.

In [ ]:
fit_res = fit_mem_2d(
    fdc_matrix=result["mat_lin"],
    fdc_time_axis=result["mat_lin_t"],
    n_components=3,
    tau_range=(0.1, 10.0),  # ns
    tau_step=0.05,
    regulator=0.1,
)

print(f"Fitting Success: {fit_res['success']}")
print(f"Q-value: {fit_res['q_value']:.4f}")

## 5. Visualize Fit and Residuals

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

im1 = ax1.imshow(np.log10(fit_res["model"] + 1), origin="lower", cmap="plasma")
ax1.set_title("2D-MEM Model")
ax1.set_xlabel("\u03c4\u2081 (bins)")
ax1.set_ylabel("\u03c4\u2082 (bins)")
plt.colorbar(im1, ax=ax1)

# Residuals calculation
weights = 1.0 / np.sqrt(np.abs(result["mat_lin"]) + 1.0)
residuals = (fit_res["model"] - result["mat_lin"]) * weights

im2 = ax2.imshow(residuals, origin="lower", cmap="RdBu", vmin=-3, vmax=3)
ax2.set_title("Weighted Residuals")
ax2.set_xlabel("\u03c4\u2081 (bins)")
ax2.set_ylabel("\u03c4\u2082 (bins)")
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()